# Experiment 3 - Encrypted Healthcare Risk Score

Evaluates a risk score over a cohort of 1000 synthetic patients under encryption, and compares its runtime and its result against the same computation in plaintext.

The score combines five measurements per patient and includes two interaction terms and one squared term, so it needs ciphertext-by-ciphertext multiplication rather than multiplication by public constants only.

The score is synthetic and built for demonstration only. It is not a clinical model and has no medical meaning.

Corresponds to Experiment 3 in the report, *Use Case: Privacy-Preserving Healthcare Analytics*.

## 1. Install and load the project


In [ ]:
import subprocess
import sys

# TenSEAL publishes real platform-tagged wheels (win_amd64 / manylinux / macosx,
# CPython 3.8-3.12), so it installs anywhere. Pinned to the version the report used.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tenseal==0.3.15", "numpy", "matplotlib"], check=True)

# OpenFHE is the opposite. Every OpenFHE distribution on PyPI is tagged
# `py3-none-any`, but the payload is Linux ELF binaries built against one
# specific CPython. The 1.4.1 line, which the report used, is:
#
#     openfhe==1.4.1.0.20.4   Ubuntu 20.04   cpython-38    requires_python >=3.8
#     openfhe==1.4.1.0.22.4   Ubuntu 22.04   cpython-310   requires_python >=3.10
#     openfhe==1.4.1.0.24.4   Ubuntu 24.04   cpython-312   requires_python >=3.12
#
# Those bounds are `>=`, so an unpinned install takes the newest wheel whose
# bound this interpreter satisfies -- which need not be built for it. On 3.10
# and 3.12 that happens to land on a valid build; on 3.9, 3.11 and 3.13 it does
# not (3.11 resolves to the cpython-310 wheel openfhe==1.5.1.0.22.4), installs
# with no warning, and fails at *import*. Pinning also keeps the OpenFHE version
# identical to the report's.
OPENFHE_WHEELS = {(3, 8): "1.4.1.0.20.4", (3, 10): "1.4.1.0.22.4", (3, 12): "1.4.1.0.24.4"}
_py = sys.version_info[:2]
_wheel = OPENFHE_WHEELS.get(_py)

if sys.platform != "linux":
    print(f"Skipping OpenFHE: no wheel has ever been published for {sys.platform!r}. "
          "The plaintext and TenSEAL columns still run.")
elif _wheel is None:
    _have = ", ".join(f"{a}.{b}" for a, b in sorted(OPENFHE_WHEELS))
    print(f"Skipping OpenFHE: no build exists for CPython {_py[0]}.{_py[1]} "
          f"(builds exist for {_have}). The plaintext and TenSEAL columns still run.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"openfhe=={_wheel}"], check=True)
    print(f"Installed openfhe=={_wheel} for CPython {_py[0]}.{_py[1]}.")

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/To2004/confidential-computing-project.git"

# Works both on a fresh Colab runtime and inside a local checkout.
if not os.path.exists("src/benchmark_harness.py"):
    if not os.path.exists("confidential-computing-project"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("confidential-computing-project")

# Absolute, so imports survive a later change of directory, and guarded so
# re-running this cell does not stack duplicate entries.
SRC = os.path.abspath("src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Write this notebook's output to its own directory, so running it does not
# overwrite the results and figures the report was built from.
import benchmark_harness as harness
import project_paths

os.makedirs("notebook_output", exist_ok=True)
project_paths.RESULTS_DIR = "notebook_output"
project_paths.FIGURES_DIR = "notebook_output"

print("working directory:", os.getcwd())
print("report uses", harness.DEFAULT_REPEATS, "repetitions per measurement")

## 2. Check which libraries loaded

OpenFHE only publishes Linux binaries, and each build targets one specific
CPython (3.8, 3.10 or 3.12). On any other platform or Python version the
install cell above skips it, so it will not be listed as available here.

Without OpenFHE the experiment still runs, with the plaintext and TenSEAL columns only.

In [ ]:
import importlib

for name in ("tenseal", "openfhe"):
    try:
        importlib.import_module(name)
        print(f"{name}: available")
    except Exception as exc:
        print(f"{name}: not available -- {exc}")

## 3. The three design decisions

Combining the primitive operations into one workload forces choices the isolated benchmarks
never have to make. Three of them determine whether the encrypted result is correct at all.

**Packing.** One *feature* per ciphertext, one *patient* per slot: five ciphertexts of 1024
slots, so a single homomorphic multiplication advances all 1000 patients at once. The
alternative - one ciphertext per patient - would multiply the operation count by a thousand.

**No division.** CKKS has no division, so min-max normalization is rewritten as an affine map
with public constants.

**Depth.** Every multiplication consumes a level from a budget fixed at setup, so the depth has
to be planned before any parameters are chosen.

In [ ]:
import notebook_tools as nt

nt.show_sources([
    ("synthetic_patients.py", "normalization_affine_terms",
     "`(x - low) / (high - low)` becomes `x * scale + offset`, because CKKS cannot divide."),
    ("risk_score_benchmark.py", "scaled_score_weights",
     "The x100 score scale is folded into each weight rather than applied at the end, which "
     "saves one whole multiplicative level."),
    ("synthetic_patients.py", "multiplicative_depth_required",
     "Four levels in total. Neither library can run that at the Experiment 1 settings: "
     "TenSEAL fails with `scale out of bounds`, and a modulus chain long enough for four "
     "levels does not fit at ring 8192 - so the ring has to double to 16384."),
])

### Why the padding *value* is part of the specification

OpenFHE requires a power-of-two batch size, so a 1000-patient cohort is padded to 1024 slots.
The cohort mean is a sum over all 1024 slots divided by 1000 - so whatever sits in those 24
padded slots lands in the answer.

Padding with each feature's range minimum is correct, because a minimum normalizes to exactly
zero and the padded slots drop out of the sum. Padding with zero is not: a raw 0 normalizes to
`-min / (max - min)`, which for systolic blood pressure is **-1.0**, not 0.

The cell below computes both, in the clear, with no encryption involved.

In [ ]:
nt.show_source(
    "risk_score_benchmark.py", "pad_features",
    note="A unit test enforces this invariant: "
         "`tests/test_synthetic_patients.py::test_padding_with_zeros_would_corrupt_the_sum`.")

In [ ]:
import numpy as np
import synthetic_patients as patients
import risk_score_benchmark as rsb

N_PATIENTS = 1000
cohort = patients.generate_cohort(n_patients=N_PATIENTS, seed=patients.DEFAULT_SEED)
n_slots = harness.next_power_of_two(N_PATIENTS)

padded_with_min = rsb.pad_features(cohort, n_slots)
padded_with_zero = {
    name: np.concatenate([np.asarray(values, dtype=float),
                          np.zeros(n_slots - len(values))])
    for name, values in cohort.items()
}

reference = patients.plaintext_risk_scores(cohort).mean()
mean_min = patients.plaintext_risk_scores(padded_with_min).sum() / N_PATIENTS
mean_zero = patients.plaintext_risk_scores(padded_with_zero).sum() / N_PATIENTS

print(f"{N_PATIENTS} patients padded to {n_slots} slots "
      f"({n_slots - N_PATIENTS} padded slots)\n")
print(f"{'reference cohort mean (unpadded)':<38}{reference:12.6f}")
print(f"{'padded with each feature minimum':<38}{mean_min:12.6f}"
      f"   shift {mean_min - reference:+.6f}")
print(f"{'padded with zeros':<38}{mean_zero:12.6f}"
      f"   shift {mean_zero - reference:+.6f}")

per_slot = (mean_zero - reference) * N_PATIENTS / (n_slots - N_PATIENTS)
print(f"\nEach zero-padded slot contributes {per_slot:+.4f} score points on a 0-100 scale.")
print("Large enough to move the answer; small enough to look plausible. That is the "
      "failure mode\nthe padding rule exists to prevent.")

## 4. Run the experiment

`REPEATS` is the number of timed repetitions per measurement. The report uses 1000 on a reserved compute node; this notebook uses fewer so it finishes in a few minutes. The numbers it prints will therefore be noisier than the ones in the report.

`--patients` sets the cohort size.

In [ ]:
REPEATS = 20

!"{sys.executable}" src/risk_score_benchmark.py --repeats {REPEATS} --warmup 5 --patients 1000 --output notebook_output/risk_score_results.json

## 5. Results


In [ ]:
import plot_results as pr
from IPython.display import Image, display

pr.apply_style()
pr.chart_risk_score(pr.load("risk_score_results.json"))
display(Image("notebook_output/chart_risk_score.png"))